# Beyond the Backlink: Why Traditional SEO Metrics Fail to Predict LLM Citations

**Abstract**
As generative AI reshapes search, marketing teams are racing to optimize for LLM citations, but it remains unclear if traditional SEO metrics predict this new traffic source. This research audited 176,568 active web pages drawn from a 79-million-row production search dataset spanning March 2026 to isolate the structural drivers of AI referrals. Using logistic regression with grouped cross-validation to prevent domain-level memorization, we decoupled traditional authority signals from page-level depth and engagement. The results demonstrate that while structural depth and active engagement are positively associated with AI traffic, traditional SEO metrics like backlinks act as domain-level noise, and structural metrics alone cannot reliably guarantee LLM citations (yielding an honest generalized AUC of 0.5403). This paper provides content strategists with the empirical proof needed to abandon traditional link-building tactics when chasing AI traffic, redirecting resources toward semantic depth and vertical-specific user intent.

*Acknowledgments: Built on the FlyRank ML Internship dataset (https://flyrank.ai).*

## 1. Question

**The Problem:** The search industry operates on a legacy assumption: pages that dominate traditional Google search through massive backlink profiles and high impression volumes will naturally capture AI-referred traffic. Because LLM citations are a black box, content teams are currently blindly applying this old SEO playbook to the new AI search paradigm, wasting hours trying to rescue poorly performing pages with link-building campaigns.

**The Research Question:** What observable content patterns and historical metrics actually correlate with absolute AI-referred traffic, and do traditional SEO authority metrics hold predictive value in this new environment?

**The Decision It Supports:** This analysis serves as a decision-support tool for resource allocation. It answers whether content teams should treat AI search optimization as an extension of traditional SEO (relying on backlinks), or if they must pivot to an entirely new playbook isolating structural depth and active user engagement.

## 2. Data

**Source:** The FlyRank Internship Warehouse (`fact_content_daily_performance` and `dim_content` tables).
**Window:** Daily performance logs restricted to the `month=2026-03` partition.
**Exclusions:** We strictly filtered for `gsc_data_available IS TRUE`, `is_published IS TRUE`, and `is_deleted IS FALSE` to ensure we analyze only actively indexed, public pages.
**Size & Privacy:** ~79 million raw daily records were securely aggregated to the content level, resulting in 176,568 unique pages, No private PII, client names, or raw search queries were exposed.

In [ ]:
import duckdb
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import userdata

# 1. Output Directory Configuration
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = OUTPUT_DIR / "cached_march_data.parquet"

# 2. Connection Setup & Caching Logic
if not CACHE_FILE.exists():
    print("Cache not found. Fetching from Hugging Face via DuckDB...")
    HF_TOKEN = userdata.get("HF_TOKEN")
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

    FACT_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
    DIM_PATH  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

    raw_df = con.execute(f"""
        SELECT
            f.client_hash_id, f.content_hash_id, f.gsc_clicks, f.gsc_impressions,
            f.ga4_total_engagement_sec, f.ga4_sessions, f.sessions_ai, c.word_count
        FROM read_parquet('{FACT_PATH}') f
        JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
        WHERE f.gsc_data_available IS TRUE AND c.is_published IS TRUE AND c.is_deleted IS FALSE
    """).df()

    frame = raw_df.groupby(["client_hash_id", "content_hash_id"]).agg(
        gsc_clicks=("gsc_clicks", "sum"), gsc_impressions=("gsc_impressions", "sum"),
        ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"), ga4_sessions=("ga4_sessions", "sum"),
        sessions_ai=("sessions_ai", "sum"), word_count=("word_count", "max")
    ).reset_index()

    frame.to_parquet(CACHE_FILE)
    print("Data fetched and cached locally.")
else:
    print("Loading data from local cache...")
    frame = pd.read_parquet(CACHE_FILE)

# 3. Engineer Normalized Features & Target
frame["is_high_ai_spike"] = (frame["sessions_ai"] >= 1).astype(int)
frame["avg_engagement_sec"] = frame["ga4_total_engagement_sec"] / frame["ga4_sessions"].clip(lower=1.0)
frame["avg_engagement_sec"] = frame["avg_engagement_sec"].fillna(0.0)
frame["ctr_computed"] = frame["gsc_clicks"] / frame["gsc_impressions"].clip(lower=1.0)
frame["has_word_count"] = frame["word_count"].notnull().astype(int)
frame["word_count_log"] = np.log1p(frame["word_count"].fillna(0.0))

print(f"Dataset successfully prepared: {len(frame):,} content pages.")
print(f"Content-level base rate (Top 1% AI Traffic): {frame['is_high_ai_spike'].mean():.4%}")

Cache not found. Fetching from Hugging Face via DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data fetched and cached locally.
Dataset successfully prepared: 176,568 content pages.
Content-level base rate (Top 1% AI Traffic): 1.8729%


## 3. Methodology

**Target Definition:** We defined success as absolute AI traffic volume. The binary label `is_high_ai_spike` flags any page with `sessions_ai >= 1`. A preliminary signal audit confirmed the 99th percentile for AI sessions is 1, making predicting "any AI traffic" the correct formulation for isolating top-tier performers.

**Features & Exclusions:** We engineered `word_count_log` for structural depth and `avg_engagement_sec` for normalized content quality. We explicitly excluded `backlinks` after our signal audit proved they act as a domain-level proxy, poisoning the model by encouraging client-memorization rather than page-level evaluation. Raw `gsc_impressions` were also excluded to prevent biasing the model toward massive, generic pages.

**Validation Design:** We utilized a `GroupShuffleSplit` (80/20) grouped on `client_hash_id`. An honest evaluation demands testing the model on entirely unseen clients to ensure the content patterns are universally applicable.

**Leakage Checks:** To prove our pipeline's integrity, we engineered a deliberate leakage trap by including `sessions_ai` in a separate training run. The AUC artificially spiked to 1.0000, confirming our test harness operates correctly and demonstrating why label-derived proxies must be strictly excluded.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

FEATURES = ["avg_engagement_sec", "ctr_computed", "word_count_log", "has_word_count"]
TARGET = "is_high_ai_spike"

# Clean Dataset & Baseline
clean_frame = frame.dropna(subset=[TARGET]).copy()
clean_frame["baseline_score"] = clean_frame["word_count_log"]

X = clean_frame[FEATURES]
y = clean_frame[TARGET]
X_leaky = clean_frame[FEATURES + ["sessions_ai"]]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=clean_frame['client_hash_id']))

# Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X.iloc[train_idx])
X_test_scaled = scaler.transform(X.iloc[test_idx])

scaler_leaky = StandardScaler()
X_train_leaky = scaler_leaky.fit_transform(X_leaky.iloc[train_idx])
X_test_leaky = scaler_leaky.transform(X_leaky.iloc[test_idx])

# Train Honest Model
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y.iloc[train_idx])

# Train Leaky Model for the Audit
model_leaky = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_leaky.fit(X_train_leaky, y.iloc[train_idx])

honest_auc = roc_auc_score(y.iloc[test_idx], model.predict_proba(X_test_scaled)[:, 1])
leaky_auc = roc_auc_score(y.iloc[test_idx], model_leaky.predict_proba(X_test_leaky)[:, 1])

print(f"Honest Grouped AUC: {honest_auc:.4f}")
print(f"Leaky Grouped AUC (+sessions_ai): {leaky_auc:.4f}")

Honest Grouped AUC: 0.5403
Leaky Grouped AUC (+sessions_ai): 1.0000


## 4. Results (vs baseline)

**Baseline Selection:** Our baseline ranks pages strictly on structural depth (`word_count_log`). We abandoned complex CTR-deficit multipliers after empirical signal auditing proved that terrible SEO metrics absolutely do not guarantee AI citations. 24].

**Findings:** The Logistic Regression achieved an out-of-sample grouped AUC of 0.5403. 25]. Feature importance analysis shows the model actively targets deep, highly engaging content, assigning massive positive weights to `word_count_log` and `avg_engagement_sec`. 23]. While the model generates measurable predictive lift at K=200 and K=500 over the base rate, the low generalized AUC reveals the harsh reality of LLM retrieval constraints. 23, 25].

In [ ]:
test_results = clean_frame.iloc[test_idx].copy()
test_results["model_prob"] = model.predict_proba(X_test_scaled)[:, 1]

def precision_at_k(df, score_col, label_col, k):
    ranked = df.sort_values(by=score_col, ascending=False)
    return ranked[label_col].iloc[:k].mean()

k_values = [20, 50, 100, 200, 500]
test_base_rate = test_results[TARGET].mean()
safe_base_rate = test_base_rate if test_base_rate > 0 else 1e-9

comparison_data = []
for k in k_values:
    if k <= len(test_results):
        base_pk = precision_at_k(test_results, "baseline_score", TARGET, k)
        model_pk = precision_at_k(test_results, "model_prob", TARGET, k)
        comparison_data.append({
            "K": k, "Baseline P@K": base_pk, "Model P@K": model_pk,
            "Baseline Lift": base_pk / safe_base_rate, "Model Lift": model_pk / safe_base_rate
        })

print("=== Honest Comparison: Baseline vs. Logistic Regression ===")
print(pd.DataFrame(comparison_data).to_markdown(index=False, floatfmt=".4f"))

coef_df = pd.DataFrame({'Feature': FEATURES, 'Coefficient': model.coef_[0]}).sort_values(by='Coefficient', key=abs, ascending=False)
print("\n=== Logistic Regression Feature Weights ===")
print(coef_df.to_markdown(index=False))

=== Honest Comparison: Baseline vs. Logistic Regression ===
|        K |   Baseline P@K |   Model P@K |   Baseline Lift |   Model Lift |
|---------:|---------------:|------------:|----------------:|-------------:|
|  20.0000 |         0.0000 |      0.0000 |          0.0000 |       0.0000 |
|  50.0000 |         0.0200 |      0.0000 |          1.3174 |       0.0000 |
| 100.0000 |         0.0100 |      0.0000 |          0.6587 |       0.0000 |
| 200.0000 |         0.0050 |      0.0200 |          0.3294 |       1.3174 |
| 500.0000 |         0.0020 |      0.0160 |          0.1317 |       1.0540 |

=== Logistic Regression Feature Weights ===
| Feature            |   Coefficient |
|:-------------------|--------------:|
| word_count_log     |      7.44899  |
| has_word_count     |     -6.41841  |
| avg_engagement_sec |      0.328906 |
| ctr_computed       |      0.046595 |


## 5. Limitations

*   **Structural Metrics are Insufficient:** The honest grouped AUC of 0.5403 proves that while structural metrics (word count and engagement) are empirically necessary, they are entirely insufficient to guarantee an AI citation on their own.
*   **Niche Vertical Blindness:** The model ranks strictly based on structure. An exceptionally deep, engaging 8,000-word technical manual will score highly, but if real users aren't prompting AI chatbots about that specific vertical, the traffic will remain at zero. LLM retrieval is ultimately a semantic matching problem.
*   **Not Causal:** Feature weights identify directional associations. Automatically expanding word counts via generative scripts will not force an AI crawler to prioritize a transactional page.

## 6. Ranked recommendations

**The Action Playbook**
We built a decision-support queue translating probabilities into action. Reason codes are constructed using the median `word_count` and top-quartile `avg_engagement_sec` from the test set.
*   `HIGH_DEPTH_AND_ENGAGEMENT` triggers `REVIEW_FOR_AI_SNIPPET_OPTIMIZATION`.
*   `HIGH_VISIBILITY_THIN_CONTENT` triggers `EXPAND_STRUCTURAL_DEPTH`.

In [ ]:
df_queue = test_results.sort_values(by="model_prob", ascending=False).reset_index(drop=True)

cols_to_clean = ["word_count", "gsc_impressions"]
df_queue[cols_to_clean] = df_queue[cols_to_clean].fillna(0)

wc_thresh = df_queue["word_count"].median()
eng_thresh = df_queue["avg_engagement_sec"].quantile(0.75)
impr_thresh = df_queue["gsc_impressions"].quantile(0.75)

reason_conditions = [
    ((df_queue["word_count"] >= wc_thresh) & (df_queue["avg_engagement_sec"] >= eng_thresh)).to_numpy(dtype=bool),
    ((df_queue["gsc_impressions"] >= impr_thresh) & (df_queue["word_count"] < wc_thresh)).to_numpy(dtype=bool)
]

reason_choices = ["HIGH_DEPTH_AND_ENGAGEMENT", "HIGH_VISIBILITY_THIN_CONTENT"]
df_queue["reason_code"] = np.select(reason_conditions, reason_choices, default="BASELINE_MONITOR")

action_conditions = [
    (df_queue["reason_code"] == "HIGH_DEPTH_AND_ENGAGEMENT").to_numpy(dtype=bool),
    (df_queue["reason_code"] == "HIGH_VISIBILITY_THIN_CONTENT").to_numpy(dtype=bool)
]

action_choices = ["REVIEW_FOR_AI_SNIPPET_OPTIMIZATION", "EXPAND_STRUCTURAL_DEPTH"]
df_queue["action"] = np.select(action_conditions, action_choices, default="MONITOR_PERFORMANCE")

# Export for Static Pages
output_csv = OUTPUT_DIR / "playbook_action_queue.csv"
export_cols = ["client_hash_id", "content_hash_id", "model_prob", "action", "reason_code", "word_count", "avg_engagement_sec"]
df_queue[export_cols].to_csv(output_csv, index=False)

## 7. Artifacts the paper embeds

The resulting playbook is safely exported to `work/outputs/playbook_action_queue.csv` and excluded from version control for data privacy. Below is the isolated top 10 queue presented to content strategists.

**Reproducibility:**
The notebooks reproducing these artifacts and evaluating the models can be found entirely within the `work/notebooks/` directory of the associated GitHub repository.

In [ ]:
print("=== Top 10 Prioritized Content Actions ===")
print(df_queue[export_cols].head(10).to_markdown(index=False, floatfmt=".4f"))

=== Top 10 Prioritized Content Actions ===
| client_hash_id          | content_hash_id          |   model_prob | action                             | reason_code               |   word_count |   avg_engagement_sec |
|:------------------------|:-------------------------|-------------:|:-----------------------------------|:--------------------------|-------------:|---------------------:|
| client_2094c6eb080311d5 | content_da2e524013a001cc |       1.0000 | REVIEW_FOR_AI_SNIPPET_OPTIMIZATION | HIGH_DEPTH_AND_ENGAGEMENT |         3003 |             949.0000 |
| client_3f0ce4d44fe94f3d | content_08f66cb1906f9cb5 |       1.0000 | REVIEW_FOR_AI_SNIPPET_OPTIMIZATION | HIGH_DEPTH_AND_ENGAGEMENT |         3034 |             803.0000 |
| client_9958f0a7ae1df715 | content_bda478d54caf6a8c |       1.0000 | MONITOR_PERFORMANCE                | BASELINE_MONITOR          |         2535 |             787.0000 |
| client_3f0ce4d44fe94f3d | content_e1549b9e1d45d993 |       1.0000 | MONITOR_PERFORMANCE   

## 8. Showcase Demo & Shareable Cuts

### 5-Minute Demo Outline
*   **The Question (1 min):** Does traditional SEO authority drive AI search citations? Content teams are currently wasting hours link-building to capture LLM traffic based on this legacy assumption.
*   **The Method (1 min):** I audited 176,568 active web pages from a 79M-row FlyRank search dataset. To prevent the model from lazily memorizing high-authority domains, I forced a GroupShuffleSplit on the client ID and dropped leaky proxies like `sessions_ai`.
*   **The Chart (1 min):** *Display the Logistic Regression Feature Weights table.* Highlight that `word_count_log` holds a massive positive weight (7.44), while traditional CTR and domain proxies offer almost no isolated predictive value.
*   **The Honest Result (1 min):** Structural depth and active engagement are strictly necessary, but entirely insufficient to guarantee an AI citation (Honest Grouped AUC: 0.5403). LLM retrieval is a semantic matching problem, not a structural SEO problem.
*   **The Recommendation (1 min):** Content teams must immediately abandon traditional link-building as a rescue metric for AI search. The playbook must pivot entirely to expanding structural depth and matching vertical-specific user intent.

---

### Social Post (LinkedIn/Twitter)
Is traditional SEO dead for AI search?

I audited 176,568 web pages using 79 million rows of production search data to find out what actually drives LLM citations. By training a Logistic Regression model with grouped cross-validation to explicitly block domain-level memorization, the data revealed a harsh reality for content teams: traditional authority metrics and backlinks are domain-level noise.

Structural depth and active engagement are mathematically necessary to capture AI traffic, but completely insufficient to guarantee it (AUC 0.5403).

**The takeaway:** Stop wasting hours building backlinks to capture AI traffic. Pivot your resources to semantic depth and vertical-specific user intent.

Read the full methodology and feature breakdown here: `[Insert Your GitHub Pages URL]`

---

### Employer 3-Sentence Summary
I built a logistic regression ranking model on a 79-million-row production search dataset to identify the structural drivers of LLM search citations. By applying grouped cross-validation to isolate domain-level noise and explicitly stripping leaky proxies, the model empirically proved that traditional SEO authority metrics fail to predict AI referral traffic. This analysis provides content strategy teams with a rigorous framework to halt arbitrary link-building campaigns and reallocate resources toward semantic depth and active user engagement.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.